In [49]:
import pandas as pd
import numpy as np
import requests
from requests.exceptions import HTTPError
import os
from dotenv import load_dotenv
from etl import extract
import datetime

load_dotenv()
access_key = os.getenv('ACCESS_KEY')

# Define API endpoint
url = 'https://www.goflightlabs.com/flights'

# Extracting data from API endpoint
flight_data_raw = extract(url, access_key)
print(flight_data_raw.shape)

Returned status code: 200
(100, 23)


In [50]:
# EDA
display(flight_data_raw.head(), flight_data_raw.info(), flight_data_raw.describe())

print('\nNumber of null values for every column feature\n')
flight_data_raw.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   hex            100 non-null    object 
 1   reg_number     100 non-null    object 
 2   flag           100 non-null    object 
 3   lat            100 non-null    float64
 4   lng            100 non-null    float64
 5   alt            100 non-null    int64  
 6   dir            100 non-null    float64
 7   speed          100 non-null    int64  
 8   v_speed        100 non-null    int64  
 9   flight_number  100 non-null    object 
 10  flight_icao    100 non-null    object 
 11  flight_iata    100 non-null    object 
 12  dep_icao       100 non-null    object 
 13  dep_iata       100 non-null    object 
 14  arr_icao       100 non-null    object 
 15  arr_iata       100 non-null    object 
 16  airline_icao   100 non-null    object 
 17  airline_iata   100 non-null    object 
 18  aircraft_ic

,hex,reg_number,flag,lat,lng,alt,dir,speed,v_speed,flight_number,...,dep_iata,arr_icao,arr_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type,squawk
0,71C391,HL8391,KR,38.900540,-7.929065,5978,238.1,531,0,921,...,ICN,LPPT,LIS,KAL,KE,B789,1761334815,en-route,adsb,NaN
1,A76516,N576DZ,US,39.613167,-91.071041,11907,129.2,998,0,188,...,ICN,KATL,ATL,DAL,DL,A359,1761334815,en-route,adsb,NaN
2,0D0EF2,XA-RFD,MX,24.877724,-100.189196,3759,176.0,316,0,105,...,MTY,MMSP,SLP,RFD,ZV,C208,1761334814,en-route,adsb,NaN
3,39348E,F-GNEO,FR,38.822458,20.743234,11328,128.0,929,0,3650,...,ORY,LGAV,ATH,TVF,TO,A20N,1761334814,en-route,adsb,NaN
4,885116,HS-THV,TH,23.905239,82.900700,11907,110.2,1006,0,925,...,MUC,VTBS,BKK,THA,TG,A359,1761334815,en-route,adsb,NaN


None

,lat,lng,alt,dir,speed,v_speed,updated
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.0,1.000000e+02
mean,25.535606,-13.607348,9310.700000,185.184000,795.470000,0.0,1.761335e+09
std,22.186302,79.988192,3206.003225,100.553628,174.281756,0.0,2.190429e-01
min,-35.535702,-127.234653,201.000000,3.800000,246.000000,0.0,1.761335e+09
25%,18.198775,-89.070720,7772.750000,110.100000,745.500000,0.0,1.761335e+09
50%,25.983777,-8.849357,10840.000000,183.200000,838.000000,0.0,1.761335e+09
75%,43.150104,72.302450,11390.250000,279.825000,900.250000,0.0,1.761335e+09
max,57.977871,148.129508,12653.000000,357.800000,1099.000000,0.0,1.761335e+09



Number of null values for every column feature



hex               0
reg_number        0
flag              0
lat               0
lng               0
alt               0
dir               0
speed             0
v_speed           0
flight_number     0
flight_icao       0
flight_iata       0
dep_icao          0
dep_iata          0
arr_icao          0
arr_iata          0
airline_icao      0
airline_iata      0
aircraft_icao     0
updated           0
status            0
type              0
squawk           96
dtype: int64

<h1>Data Cleaning</h1>

<li>Creating a copy of the raw flight data and applying data cleaning</li>
<li>Replacing missing values in "squawk" column with "unknown"</li>
<li>Converting "updated" values from timestamp to datetime</li>

In [ ]:
# Copying raw flight data
flight_data_raw_copy = flight_data_raw.copy()

# Replacing missing values in 'squawk' column with 'Unknown'
flight_data_raw_copy['squawk'] = flight_data_raw_copy['squawk'].fillna('Unknown')

# Converting 'updated' values from timestamp to datetime
flight_data_raw_copy['updated'] = flight_data_raw_copy['updated'].apply(lambda x: datetime.datetime.fromtimestamp(x))
display(flight_data_raw_copy.head())

print('\nNumber of null values for every column feature\n')
flight_data_raw_copy.isnull().sum()


,hex,reg_number,flag,lat,lng,alt,dir,speed,v_speed,flight_number,...,dep_iata,arr_icao,arr_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type,squawk
0,71C391,HL8391,KR,38.900540,-7.929065,5978,238.1,531,0,921,...,ICN,LPPT,LIS,KAL,KE,B789,2025-10-24 15:40:15,en-route,adsb,Unknown
1,A76516,N576DZ,US,39.613167,-91.071041,11907,129.2,998,0,188,...,ICN,KATL,ATL,DAL,DL,A359,2025-10-24 15:40:15,en-route,adsb,Unknown
2,0D0EF2,XA-RFD,MX,24.877724,-100.189196,3759,176.0,316,0,105,...,MTY,MMSP,SLP,RFD,ZV,C208,2025-10-24 15:40:14,en-route,adsb,Unknown
3,39348E,F-GNEO,FR,38.822458,20.743234,11328,128.0,929,0,3650,...,ORY,LGAV,ATH,TVF,TO,A20N,2025-10-24 15:40:14,en-route,adsb,Unknown
4,885116,HS-THV,TH,23.905239,82.900700,11907,110.2,1006,0,925,...,MUC,VTBS,BKK,THA,TG,A359,2025-10-24 15:40:15,en-route,adsb,Unknown



Number of null values for every column feature



hex              0
reg_number       0
flag             0
lat              0
lng              0
alt              0
dir              0
speed            0
v_speed          0
flight_number    0
flight_icao      0
flight_iata      0
dep_icao         0
dep_iata         0
arr_icao         0
arr_iata         0
airline_icao     0
airline_iata     0
aircraft_icao    0
updated          0
status           0
type             0
squawk           0
dtype: int64